# Control Trainer

> Fill in a module description here

In [ ]:
#| default_exp trainers.world_model

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math
import torch
import os
import torch.nn as nn
import torchvision.utils as vutils
import wandb
import hydra
from pathlib import Path
from omegaconf import DictConfig
from einops import rearrange

from c3jepa_wm.utils.checkpointer import RetrospectiveCheckpointer


In [ ]:
#| export
class BaseTrainer:
    def __init__(self, 
                 data_module, 
                 device, 
                 slurm_jobid= None, 
                 lr=1e-4, 
                 epochs=100, 
                 project_name="c3jepa_wm",
                 ckp_dir= "checkpoints",
                 save_dir= "outputs"):
        
        self.data_module = data_module
        
        self.device = device
        
        self.data_module.setup()
        self.train_loader = self.data_module.train_dataloader()
        self.val_loader = self.data_module.val_dataloader()

        self.slurm_jobid = slurm_jobid if slurm_jobid else "default_job"
        self.lr = lr
        self.epochs = epochs
        self.project_name = project_name
        self.ckp_dir = ckp_dir
        self.save_dir = save_dir

    def init_optimizer(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr)
    
    def train_epoch(self, epoch):
        raise NotImplementedError("train_epoch method must be implemented by subclasses.")   

    def validate_epoch(self, epoch):
        raise NotImplementedError("validate method must be implemented by subclasses.")
    

In [ ]:
#| export
class WMTrainer(BaseTrainer):
    def __init__(self, data_module, model, device, history_size, num_preds, lambda_sigreg, lambda_pow, lambda_value, lambda_quality, lambda_send, **kwargs):
        super().__init__(
            data_module= data_module,
            device= device, 
            **kwargs)
        
        self.history_size = history_size
        self.num_preds = num_preds

        self.lambda_sigreg = lambda_sigreg
        self.lambda_pow = lambda_pow
        self.lambda_value = lambda_value
        self.lambda_quality = lambda_quality
        self.lambda_send = lambda_send

        self.vqvae = model["vqvae"].to(device)
        self.power_net = model["power_net"].to(device)
        self.jepa = model["jepa"].to(device)

        self.optimizer = self.init_optimizer()
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=5
        )
        
        self.save_dir = Path(hydra.utils.to_absolute_path(self.save_dir))
        self.save_dir.mkdir(exist_ok=True, parents=True)

        self.ck_pointer = RetrospectiveCheckpointer(project_name= self.project_name,
                                                    ckp_dir= self.ckp_dir,
                                                    slurm_jobid= self.slurm_jobid,
                                                    agent_id= "vqvae_trainer",
                                                    rank= 0, 
                                                    n_best=3)

        # Create output directories for visual inspection
        os.makedirs(os.path.join(self.save_dir, "Reconstructions"), exist_ok=True)


    @torch.no_grad()
    def get_msg_indices(self, sender_pov_seq):
        """Helper function to get message indices from the pretrained VQ-VAE for a given sender POV sequence.
        Args:
            sender_pov_seq: Tensor of shape (B, T, C, H, W) representing the sender's point-of-view image sequence.
        Returns:
            msg_indices: Tensor of shape (B*T, 49) 
        """
        B, T, C, H, W = sender_pov_seq.shape
        sender_pov_flat = rearrange(sender_pov_seq.to(self.device), "b t c h w -> (b t) c h w")  # (B*T, C, H, W)
        msg_indices_flat = self.vqvae.get_message_indices(sender_pov_flat)  # (B*T, 7, 7)
        msg_indices = rearrange(msg_indices_flat, "b h w -> b (h w)", b=B*T)   # (B*T, 49)
        return msg_indices


    def train_epoch(self, epoch):
        for batch in self.train_loader:

            # self.optimizer.zero_grad() # comment for now until we still down optimizer choice.
            # 1. Get message indices from pretrained VQ-VAE
            
            msg_indices = self.get_msg_indices(batch["sender_pov_seq"])   # (B*T, 49)
            csi_flat = rearrange(batch["sender_csi"].to(self.device), "b t n d -> (b t) n d")  # (B*T, N, D)
            schedule, power = self.power_net(msg_indices, csi_flat) # power and schedule pred     # (B*T, n, 1)       
            
            received_msg  = self.channel(schedule, power, msg_indices, csi_flat) # channel output (B*T, 49)
            received_msg = rearrange(received_msg, "(b t) msg_dim -> b t msg_dim", msg_dim=49) # (B, T, 49)

            # 2. Encode receiver history with JEPA encoder
            output = self.model.encode(batch)
            
            # 3. Forward through world model predictor
            ctx_len = self.history_size
            n_preds = self.num_preds
            lambd = self.lambda_sigreg
            
            emb = output["emb"]  # (B, T+1, D)
            act_emb = output["act_emb"] # (B, T, D_act)

            ctx_emb = emb[:, :ctx_len]
            ctx_act = act_emb[:, : ctx_len]
            ctx_msg = received_msg[:, :ctx_len] # (B, ctx_len, msg_dim)

            tgt_emb = emb[:, n_preds:] # label
            pred_emb = self.model.predict(ctx_emb, ctx_act, ctx_msg) # pred           

            # 4. Compute losses
            output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, lambd)
            # output["pred_loss"] = (pred_emb - tgt_emb).pow(2).mean()
            # output["sigreg_loss"]= self.sigreg(emb.transpose(0, 1))
            # output["jepa_loss"] = output["pred_loss"] + lambd * output["sigreg_loss"] 

            output['tx_loss'] = self.power_net.loss_fn(self.model, 
                                                       ctx_len,
                                                       ctx_emb,
                                                       ctx_act,
                                                       msg_indices,
                                                       tgt_emb,
                                                       schedule,
                                                       power,
                                                       output["pred_loss"]
                                                       )

            output['jepa_loss'].backward(retain_graph=True)
            output['tx_loss'].backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(self.power_net.parameters(), max_norm=1.0)

            self.jepa_optimizer.step()
            self.power_optimizer.step()

    @torch.no_grad()
    def channel(self, schedule, power, msg_indices, csi, code_book_size= 128):
        """Applies channel effect: 1- Binarization, 2-BPSK modulation, 3- Power scaling, 4- AWGN noise addition, 5- Demodulation, 6- De-binarization
        Args:
            schedule: (B*T, n, 1)
            power: (B*T, n, 1)
            msg_indices: (B*T, msg_dim)
            csi: (B*T, n, 1)
        Returns:
            recovered_indices: (B*T, msg_dim)
        """
        if schedule is None or power is None:
            return msg_indices  # No channel effects, return original message indices
        else:
            symbol_length = int(math.log2(code_book_size)) # Number of bits needed to represent codebook size
            schedule = rearrange(schedule, "b n d -> (b n) d", d= 1)  # (B*T*n, d)
            power = rearrange(power, "b n d -> (b n) d", d= 1)  # (B*T*n, d)
            csi = rearrange(csi, "b n d -> (b n) d", d= 1)  # (B*T*n, d)

            # Simulate channel effects
            # Binarize message indices
            batch_size, msg_dim = msg_indices.shape
            msg_bits = torch.zeros(batch_size, msg_dim * symbol_length, device=self.device)  # Assuming 7 bits per index
            for i in range(msg_dim):
                for b in range(symbol_length):
                    msg_bits[:, i * symbol_length + b] = (msg_indices[:, i] >> b) & 1
            # BPSK modulation
            modulated = 2 * msg_bits - 1  # Map 0 -> -1, 1 -> +1
            # Apply power scaling
            power = power.unsqueeze(-1)  # (B, T, 1)
            modulated = modulated * torch.sqrt(power)  # Scale by sqrt of power
            # Add AWGN noise
            channel_noise = torch.randn_like(modulated) * 0.1  # Adjust noise power as needed
            received = modulated*csi + channel_noise
            # Demodulation (assuming perfect synchronization and channel estimation)
            demodulated = (received.real > 0).float()  # Simple thresholding for BPSK
            # De-binarization 
            recovered_indices = torch.zeros(batch_size, msg_dim, device=self.device, dtype=torch.long)
            for i in range(msg_dim):
                for b in range(symbol_length):
                    recovered_indices[:, i] += (demodulated[:, i * symbol_length + b] > 0).long() << b
            return recovered_indices
            

    def validate_epoch(self, epoch):
        pass
        # self.model.eval()
        # total_loss = 0.0
        # with torch.no_grad():
        #     for batch_idx, batch in enumerate(self.val_loader):
        #         batch = {k: v.to(self.device) for k, v in batch.items()}
        #         loss = self.model.compute_loss(batch)
        #         total_loss += loss.item()

        #         # Log reconstructions for the first batch of each epoch
        #         if batch_idx == 0:
        #             recon_images = self.model.reconstruct(batch)
        #             recon_grid = vutils.make_grid(recon_images.cpu(), nrow=4, normalize=True)
        #             wandb.log({"reconstructions": [wandb.Image(recon_grid)]}, step=epoch)

        # avg_loss = total_loss / len(self.val_loader)
        # print(f"Epoch [{epoch}/{self.epochs}] - Val Loss: {avg_loss:.4f}")
        # wandb.log({"val_loss": avg_loss}, step=epoch)

        # # Step the learning rate scheduler
        # self.scheduler.step(avg_loss)

        # # Checkpointing based on validation loss
        # self.ck_pointer.save_checkpoint(model_state=self.model.state_dict(),
        #                                 optimizer_state=self.optimizer.state_dict(),
        #                                 epoch=epoch,
        #                                 val_loss=avg_loss)
        




In [8]:
import math
int(math.log2(128))

7

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()